# model-archival — One-time Setup

Run this notebook **once** on a new Colab session to:
1. Verify your Drive has enough space
2. Test your HuggingFace token
3. Clone or copy the project files to the right place
4. Check which gated models your token can access

After setup, open and run **`archiver.ipynb`** for actual downloads.

---
## Step 1 — Mount Drive and check space

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f"Google Drive quota:")
print(f"  Total : {total / 1024**4:.2f} TB")
print(f"  Used  : {used  / 1024**4:.2f} TB  ({used / total * 100:.1f}%)")
print(f"  Free  : {free  / 1024**4:.2f} TB")

if free < 500 * 1024**3:
    print("\n⚠  Less than 500 GB free — check Drive storage before downloading large models")
else:
    print("\n✓ Sufficient space available")

---
## Step 2 — Configure HuggingFace token

### How to add the token as a Colab Secret (recommended)

1. Click the **🔑 key icon** in the left sidebar of Colab
2. Click **+ Add new secret**
3. Name: `HF_TOKEN`
4. Value: your token from https://huggingface.co/settings/tokens
5. Toggle **Notebook access** to ON
6. Re-run this cell

Your token is stored securely in your Google account and is never visible in the notebook output.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '--upgrade', 'huggingface_hub'])

from google.colab import userdata
from huggingface_hub import HfApi, whoami

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if HF_TOKEN:
    try:
        info = whoami(token=HF_TOKEN)
        print(f"✓ Token valid — logged in as: {info['name']}")
    except Exception as e:
        print(f"✗ Token invalid: {e}")
        print("  Get a new token at https://huggingface.co/settings/tokens")
        HF_TOKEN = None
else:
    print("⚠  No HF_TOKEN found in Colab Secrets")
    print("   Token-free models (Tier C, most of Tier D, DeepSeek, Qwen, Phi) will still work.")
    print("   Gated models (Llama, Gemma, Mistral Large) require a token.")

---
## Step 3 — Copy project files to Drive (run once)

This makes the downloader library and registry available across sessions without re-cloning.

**Option A** — Clone from GitHub (recommended if repo is public):

In [ ]:
import subprocess
from pathlib import Path

REPO_URL   = 'https://github.com/mishra8038/model-archival.git'
CLONE_PATH = '/content/model-archival'
DRIVE_COPY = '/content/drive/MyDrive/model-archival'

# Clone to Colab's ephemeral /content
if not Path(CLONE_PATH).exists():
    subprocess.check_call(['git', 'clone', '--depth=1', REPO_URL, CLONE_PATH])
    print(f'✓ Cloned to {CLONE_PATH}')
else:
    subprocess.check_call(['git', '-C', CLONE_PATH, 'pull', '--ff-only'])
    print(f'✓ Updated {CLONE_PATH}')

# Copy the essential files to Drive so they persist across sessions
import shutil
src = Path(CLONE_PATH)
dst = Path(DRIVE_COPY)
for sub in ['colab/lib', 'local/config']:
    (dst / sub).mkdir(parents=True, exist_ok=True)
    for f in (src / sub).iterdir():
        if f.is_file():
            shutil.copy2(f, dst / sub / f.name)
            print(f'  copied: {sub}/{f.name}')

print(f'\n✓ Files persisted to {DRIVE_COPY}')
print('  Future sessions can load from Drive without re-cloning.')

---
## Step 4 — Check gated model access

In [ ]:
import yaml
from huggingface_hub import HfApi
from pathlib import Path

# Load registry
registry_path = Path('/content/drive/MyDrive/model-archival/local/config/registry.yaml')
if not registry_path.exists():
    registry_path = Path('/content/model-archival/local/config/registry.yaml')

with open(registry_path) as f:
    registry = yaml.safe_load(f)

gated = [m for m in registry['models'] if m.get('requires_auth')]
api = HfApi()

print(f'Checking access to {len(gated)} gated models...\n')

accessible = []
blocked = []

for m in gated:
    try:
        api.repo_info(repo_id=m['id'], token=HF_TOKEN)
        print(f'  ✓ {m["id"]}')
        accessible.append(m['id'])
    except Exception as e:
        err = str(e)
        if '401' in err or '403' in err or 'gated' in err.lower():
            print(f'  ✗ {m["id"]}  → access denied (accept licence at huggingface.co)')
        else:
            print(f'  ? {m["id"]}  → {e}')
        blocked.append(m['id'])

print(f'\n{len(accessible)} accessible  |  {len(blocked)} blocked')
if blocked:
    print()
    print('To unblock, visit each model page on huggingface.co and accept the licence terms.')
    print('See local/docs/HF-TOKEN-GUIDE.md for per-model instructions.')

---
## Setup complete

Open **`archiver.ipynb`** to start downloading.

**Quick tips:**
- Set `TIERS = ['C', 'D']` first — these are the smaller models (< 100 GB) and will complete in one session
- Tier A (raw BF16 70B+ models) = 140–700 GB each — plan for multiple sessions
- Enable **background execution** in Runtime menu (Pro+ feature) before closing the browser
- The session keeps a `run_state.json` on Drive — each new session resumes automatically